<a href="https://colab.research.google.com/github/qubit55/clojupyter-playground/blob/main/embeddings_clojure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# First install Clojupyter kernel

In [ ]:
!wget https://raw.githubusercontent.com/qubit55/clojupyter_colab_setup/refs/heads/main/install_clojure_kernel.sh
!chmod +x install_clojure_kernel.sh
!./install_clojure_kernel.sh

# Once the installation of Clojupyter is complete

* Go to Runtime -> Change runtime type.
* In the "Runtime type" dropdown, select Clojure IPC.
* Wait for the kernel to initialize. The RAM and disk usage chart in the upper-right corner will confirm when it’s ready.
* Now you can start executing the cells down below

In [10]:
(require '[clojupyter.misc.helper :as helper]
         '[clojupyter.display :as display])

(helper/add-dependencies '[com.knuddels/jtokkit "1.1.0"])
(helper/add-dependencies '[criterium "0.4.6"])
(helper/add-dependencies '[uncomplicate/deep-diamond "0.31.0"])
(helper/add-dependencies '[uncomplicate/neanderthal "0.53.2"])
(helper/add-dependencies '[uncomplicate/fluokitten "0.10.0"])

uncomplicate/fluokitten 0.10.0 org.clojure/clojure 1.11.3 org.clojure/clojure 1.11.3 org.clojure/core.specs.alpha 0.2.62 org.clojure/spec.alpha 0.3.218 org.clojure/spec.alpha 0.3.218 org.clojure/core.specs.alpha 0.2.62

In [13]:
(import '[com.knuddels.jtokkit Encodings]
        '[com.knuddels.jtokkit.api EncodingType IntArrayList])

(require '[clojure.repl :refer [doc]]
         '[criterium.core :refer [quick-bench]]
         '[uncomplicate.fluokitten.core :refer [fmap!]]
         '[uncomplicate.commons.core :refer [with-release]]
         '[uncomplicate.neanderthal
           [native :refer [dv dge fge dv fv iv]]
           [core :refer [mv! copy! dim entry row rows col cols submatrix subvector ncols mrows transfer! transfer]]
           [random :refer [rand-uniform!]]])

nil

In [14]:
(defn pluck-embeddings
 [tok-ids emb-mat]
 (let
  [emb-mat-subset (fge (count tok-ids) (ncols emb-mat))]
  (loop [i 0 tok-id tok-ids]
   (if (< i (mrows emb-mat-subset))
    (do
      (transfer! (row emb-mat (first tok-id)) (row emb-mat-subset i))
      (recur (inc i) (rest tok-id)))))
   emb-mat-subset))

 (defn pluck-embeddings-2
  [tok-ids emb-mat]
  (let
    [emb-mat-subset (fge (count tok-ids) (ncols emb-mat))]
    (loop [i 0 tok-id tok-ids]
    (if (< i (mrows emb-mat-subset))
      (do
        (copy! (row emb-mat (first tok-id)) (row emb-mat-subset i))
        (recur (inc i) (rest tok-id)))))
    emb-mat-subset))

(defn test-pluck-embeddings
 [pluck-embeddings tok-ids emb-mat]
 (with-release
  [emb-subset (pluck-embeddings tok-ids emb-mat)]))

#'user/test-pluck-embeddings

In [15]:
(def emb-mat (fge 4 3 {:layout :row}))
(rand-uniform! emb-mat)
(println emb-mat)

#RealGEMatrix[float, mxn:4x3, layout:row]
   ▤       ↓       ↓       ↓       ┓    
   →       0.43    0.96    0.88         
   →       0.17    0.31    0.50         
   →       0.86    0.86    0.67         
   →       0.57    0.93    0.27         
   ┗                               ┛    



nil

In [16]:
(print (pluck-embeddings [0 3 2] emb-mat))

#RealGEMatrix[float, mxn:3x3, layout:column]
   ▥       ↓       ↓       ↓       ┓    
   →       0.43    0.96    0.88         
   →       0.57    0.93    0.27         
   →       0.86    0.86    0.67         
   ┗                               ┛    


nil

In [30]:
(print (pluck-embeddings-2 [0 3 2] emb-mat))

#RealGEMatrix[float, mxn:3x3, layout:column]
   ▥       ↓       ↓       ↓       ┓    
   →       0.43    0.96    0.88         
   →       0.57    0.93    0.27         
   →       0.86    0.86    0.67         
   ┗                               ┛    


nil

In [32]:
(print emb-mat) ;; Is emb-mat still intact?

#RealGEMatrix[float, mxn:4x3, layout:row]
   ▤       ↓       ↓       ↓       ┓    
   →       0.43    0.96    0.88         
   →       0.17    0.31    0.50         
   →       0.86    0.86    0.67         
   →       0.57    0.93    0.27         
   ┗                               ┛    


nil

In [18]:
(def emb-mat-large (fge 50000 1000 {:layout :row}))
(rand-uniform! emb-mat-large)

#RealGEMatrix[float, mxn:50000x1000, layout:row]

Randomly generate arrays of token ids of various lengths for testing.

In [19]:
(def toks-5 (vec (map (fn [_] (rand-int (mrows emb-mat-large))) (range 5))))
(println toks-5)

(def toks-15 (vec (map (fn [_] (rand-int (mrows emb-mat-large))) (range 15))))
(println toks-15)

(def toks-100 (vec (map (fn [_] (rand-int (mrows emb-mat-large))) (range 100))))
(println toks-100)

(def toks-1000 (vec (map (fn [_] (rand-int (mrows emb-mat-large))) (range 1000))))
(println toks-1000)

(def toks-5000 (vec (map (fn [_] (rand-int (mrows emb-mat-large))) (range 5000))))
(println toks-5000)

[4599 4310 41446 26529 20686]
[43849 13394 42770 49132 14527 24980 39860 21550 8925 37084 28677 39364 40863 16034 10034]
[5888 37615 43047 13719 29356 48060 27009 19143 27677 32172 48408 20173 30433 15411 49544 43835 46097 30153 6343 25010 42809 25096 20054 26978 29158 2749 2911 30745 17383 44578 448 44421 5156 3131 44557 7642 23676 30595 15630 34991 19210 23594 9135 6680 12389 44784 1874 24411 14588 32803 45787 5204 23555 33490 32683 36290 21054 26918 21552 9671 3770 36778 1224 45850 44969 40885 25041 2404 8464 4209 961 44563 18362 28698 36 31774 27706 7402 8560 46349 26707 33903 44362 7710 36508 1928 35686 28514 47625 34139 21887 26974 33108 3580 28795 13079 45245 6020 47994 15087]
[41212 6546 8504 6665 27438 19114 21388 34348 14662 33535 2666 29618 35670 30345 37580 19837 45519 20391 39731 27499 49684 12581 31903 17831 39030 231 31500 20649 1741 13708 12410 13541 31451 14653 36885 9957 7653 4980 5513 47584 39220 42045 10760 19511 34328 39922 49698 33105 40432 45539 4511 19294 36056 

nil

In [20]:
(quick-bench (test-pluck-embeddings pluck-embeddings toks-5 emb-mat-large))

Evaluation count : 18 in 6 samples of 3 calls.
             Execution time mean : 45.985428 ms
    Execution time std-deviation : 10.609746 ms
   Execution time lower quantile : 33.575763 ms ( 2.5%)
   Execution time upper quantile : 56.815812 ms (97.5%)
                   Overhead used : 11.959738 ns


nil

In [22]:
(quick-bench (test-pluck-embeddings pluck-embeddings toks-15 emb-mat-large))

Evaluation count : 6 in 6 samples of 1 calls.
             Execution time mean : 131.271488 ms
    Execution time std-deviation : 22.996324 ms
   Execution time lower quantile : 102.230218 ms ( 2.5%)
   Execution time upper quantile : 153.397515 ms (97.5%)
                   Overhead used : 11.959738 ns


nil

In [21]:
(quick-bench (test-pluck-embeddings pluck-embeddings toks-100 emb-mat-large))

Evaluation count : 6 in 6 samples of 1 calls.
             Execution time mean : 993.859547 ms
    Execution time std-deviation : 530.261721 ms
   Execution time lower quantile : 676.943028 ms ( 2.5%)
   Execution time upper quantile : 1.898385 sec (97.5%)
                   Overhead used : 11.959738 ns

Found 1 outliers in 6 samples (16.6667 %)
	low-severe	 1 (16.6667 %)
 Variance from outliers : 82.6980 % Variance is severely inflated by outliers


nil

In [23]:
(quick-bench (test-pluck-embeddings pluck-embeddings toks-1000 emb-mat-large))

Evaluation count : 6 in 6 samples of 1 calls.
             Execution time mean : 7.523832 sec
    Execution time std-deviation : 617.172297 ms
   Execution time lower quantile : 6.876759 sec ( 2.5%)
   Execution time upper quantile : 8.397794 sec (97.5%)
                   Overhead used : 11.959738 ns


nil

In [24]:
(quick-bench (test-pluck-embeddings pluck-embeddings toks-5000 emb-mat-large))

Evaluation count : 6 in 6 samples of 1 calls.
             Execution time mean : 36.804169 sec
    Execution time std-deviation : 403.507972 ms
   Execution time lower quantile : 36.528748 sec ( 2.5%)
   Execution time upper quantile : 37.496309 sec (97.5%)
                   Overhead used : 11.959738 ns

Found 1 outliers in 6 samples (16.6667 %)
	low-severe	 1 (16.6667 %)
 Variance from outliers : 13.8889 % Variance is moderately inflated by outliers


nil

In [25]:
(quick-bench (test-pluck-embeddings pluck-embeddings-2 toks-5 emb-mat-large))

Evaluation count : 49266 in 6 samples of 8211 calls.
             Execution time mean : 18.693710 µs
    Execution time std-deviation : 1.280104 µs
   Execution time lower quantile : 17.573807 µs ( 2.5%)
   Execution time upper quantile : 20.284050 µs (97.5%)
                   Overhead used : 11.959738 ns


nil

In [26]:
(quick-bench (test-pluck-embeddings pluck-embeddings-2 toks-15 emb-mat-large))

Evaluation count : 11898 in 6 samples of 1983 calls.
             Execution time mean : 42.920752 µs
    Execution time std-deviation : 6.738228 µs
   Execution time lower quantile : 37.992651 µs ( 2.5%)
   Execution time upper quantile : 54.161990 µs (97.5%)
                   Overhead used : 11.959738 ns

Found 1 outliers in 6 samples (16.6667 %)
	low-severe	 1 (16.6667 %)
 Variance from outliers : 47.3585 % Variance is moderately inflated by outliers


nil

In [27]:
(quick-bench (test-pluck-embeddings pluck-embeddings-2 toks-100 emb-mat-large))

Evaluation count : 2160 in 6 samples of 360 calls.
             Execution time mean : 287.025337 µs
    Execution time std-deviation : 12.421476 µs
   Execution time lower quantile : 279.890508 µs ( 2.5%)
   Execution time upper quantile : 307.683278 µs (97.5%)
                   Overhead used : 11.959738 ns

Found 1 outliers in 6 samples (16.6667 %)
	low-severe	 1 (16.6667 %)
 Variance from outliers : 13.8889 % Variance is moderately inflated by outliers


nil

In [28]:
(quick-bench (test-pluck-embeddings pluck-embeddings-2 toks-1000 emb-mat-large))

Evaluation count : 102 in 6 samples of 17 calls.
             Execution time mean : 6.733377 ms
    Execution time std-deviation : 1.171201 ms
   Execution time lower quantile : 5.319742 ms ( 2.5%)
   Execution time upper quantile : 8.078477 ms (97.5%)
                   Overhead used : 11.959738 ns


nil

In [29]:
(quick-bench (test-pluck-embeddings pluck-embeddings-2 toks-5000 emb-mat-large))

Evaluation count : 18 in 6 samples of 3 calls.
             Execution time mean : 38.212981 ms
    Execution time std-deviation : 5.300530 ms
   Execution time lower quantile : 34.852779 ms ( 2.5%)
   Execution time upper quantile : 47.292823 ms (97.5%)
                   Overhead used : 11.959738 ns

Found 1 outliers in 6 samples (16.6667 %)
	low-severe	 1 (16.6667 %)
 Variance from outliers : 31.8295 % Variance is moderately inflated by outliers


nil